In [4]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

print("=== Step 1: Reading CICIDS2017 Dataset ===")
file_path = "/kaggle/input/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed/cicids2017_cleaned.csv"
df = pd.read_csv(file_path)
print(f"✅ Loaded dataset shape: {df.shape}")

# Clean column names
df.columns = df.columns.str.strip()

target_col = 'Attack Type'
print(f"\nLabel Breakdown in '{target_col}':")
print(df[target_col].value_counts())

# Step 2: Binary Target Creation (0 = Normal Traffic, 1 = Attack)
print("\n=== Step 2: Creating Binary Target (0: Normal, 1: Attack) ===")
df['Binary_Target'] = df[target_col].apply(lambda x: 0 if 'normal' in str(x).lower() else 1)

print("Binary Class Breakdown:")
print(df['Binary_Target'].value_counts())

# Step 3: Feature Preparation
# Drop target columns and any non-numeric/identifier columns
drop_cols = [target_col, 'Binary_Target', 'Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df['Binary_Target']

# Handle any remaining object/string columns in X if present
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

# Clean inf/nan values
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"\nFinal Feature Matrix Shape: {X.shape}")

# Step 4: Train/Test Split (80% Train, 20% Test)
print("\n=== Step 3: Train/Test Split ===")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

# Step 5: Feature Scaling
print("\n=== Step 4: Feature Scaling ===")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 6: Train XGBoost Model on CPU
print("\n=== Step 5: Training XGBoost Model on CPU ===")
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    n_jobs=-1,  # Uses all Kaggle CPU cores
    random_state=42
)
model.fit(X_train_scaled, y_train)

# Step 7: Model Evaluation
print("\n=== Step 6: Model Evaluation ===")
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"🎯 Model Accuracy: {acc * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal Traffic', 'Attack Traffic']))

# Step 8: Save Model Artifacts
print("\n=== Step 7: Saving Artifacts ===")
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

joblib.dump(model, "/kaggle/working/artifacts/cicids2017_xgboost_model.joblib")
joblib.dump(scaler, "/kaggle/working/artifacts/cicids2017_scaler.joblib")
joblib.dump(feature_cols, "/kaggle/working/artifacts/cicids2017_features.joblib")

print("🎉 SUCCESS! Saved 3 artifact files to '/kaggle/working/artifacts/':")
print(" 1. cicids2017_xgboost_model.joblib")
print(" 2. cicids2017_scaler.joblib")
print(" 3. cicids2017_features.joblib")

=== Step 1: Reading CICIDS2017 Dataset ===
✅ Loaded dataset shape: (2520751, 53)

Label Breakdown in 'Attack Type':
Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64

=== Step 2: Creating Binary Target (0: Normal, 1: Attack) ===
Binary Class Breakdown:
Binary_Target
0    2095057
1     425694
Name: count, dtype: int64

Final Feature Matrix Shape: (2520751, 52)

=== Step 3: Train/Test Split ===
Train Shape: (2016600, 52), Test Shape: (504151, 52)

=== Step 4: Feature Scaling ===

=== Step 5: Training XGBoost Model on CPU ===

=== Step 6: Model Evaluation ===
🎯 Model Accuracy: 99.89%

Classification Report:
                precision    recall  f1-score   support

Normal Traffic       1.00      1.00      1.00    419012
Attack Traffic       1.00      1.00      1.00     85139

      accuracy                           1.00   

In [5]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier

print("=== Step 1: Loading UNSW-NB15 Parquet Files ===")
train_path = "/kaggle/input/datasets/dhoogla/unswnb15/UNSW_NB15_training-set.parquet"
test_path = "/kaggle/input/datasets/dhoogla/unswnb15/UNSW_NB15_testing-set.parquet"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

print(f"✅ Loaded Train set shape: {train_df.shape}")
print(f"✅ Loaded Test set shape:  {test_df.shape}")

# Clean column names
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

# Target Label Column
target_col = 'label' if 'label' in train_df.columns else train_df.columns[-1]

print(f"\nTarget Column: '{target_col}'")
print("Train Label Breakdown:")
print(train_df[target_col].value_counts())

# Drop non-feature columns
drop_cols = [target_col, 'id', 'ID', 'attack_cat', 'Attack_Cat', 'srcip', 'dstip', 'sport', 'dsport']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].astype(int)

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].astype(int)

# Categorical & Category Feature Encoding (convert category -> str -> label encoding)
encoders = {}
for col in X_train.columns:
    if X_train[col].dtype.name in ['category', 'object']:
        le = LabelEncoder()
        # Convert to string and fit label encoder across both train & test sets
        combined = pd.concat([X_train[col].astype(str), X_test[col].astype(str)])
        le.fit(combined)
        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))
        encoders[col] = le

# Handle inf/nan values on numeric matrices
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"\nFinal Feature Matrix Shape: {X_train.shape}")

# Step 2: Feature Scaling
print("\n=== Step 2: Feature Scaling ===")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 3: Model Training on CPU
print("\n=== Step 3: Training XGBoost Model on CPU ===")
model_unsw = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    n_jobs=-1,
    random_state=42
)
model_unsw.fit(X_train_scaled, y_train)

# Step 4: Model Evaluation
print("\n=== Step 4: Model Evaluation ===")
y_pred = model_unsw.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"🎯 UNSW-NB15 Model Accuracy: {acc * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Attack (1)']))

# Step 5: Save Model Artifacts
print("\n=== Step 5: Saving Artifacts ===")
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

joblib.dump(model_unsw, "/kaggle/working/artifacts/unsw_nb15_xgboost_model.joblib")
joblib.dump(scaler, "/kaggle/working/artifacts/unsw_nb15_scaler.joblib")
joblib.dump(encoders, "/kaggle/working/artifacts/unsw_nb15_encoders.joblib")
joblib.dump(feature_cols, "/kaggle/working/artifacts/unsw_nb15_features.joblib")

print("\n🎉 SUCCESS! Saved all UNSW-NB15 artifacts to '/kaggle/working/artifacts/':")
print(" 1. unsw_nb15_xgboost_model.joblib")
print(" 2. unsw_nb15_scaler.joblib")
print(" 3. unsw_nb15_encoders.joblib")
print(" 4. unsw_nb15_features.joblib")

=== Step 1: Loading UNSW-NB15 Parquet Files ===
✅ Loaded Train set shape: (175341, 36)
✅ Loaded Test set shape:  (82332, 36)

Target Column: 'label'
Train Label Breakdown:
label
1    119341
0     56000
Name: count, dtype: int64

Final Feature Matrix Shape: (175341, 34)

=== Step 2: Feature Scaling ===

=== Step 3: Training XGBoost Model on CPU ===

=== Step 4: Model Evaluation ===
🎯 UNSW-NB15 Model Accuracy: 84.73%

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.97      0.68      0.80     37000
  Attack (1)       0.79      0.98      0.88     45332

    accuracy                           0.85     82332
   macro avg       0.88      0.83      0.84     82332
weighted avg       0.87      0.85      0.84     82332


=== Step 5: Saving Artifacts ===

🎉 SUCCESS! Saved all UNSW-NB15 artifacts to '/kaggle/working/artifacts/':
 1. unsw_nb15_xgboost_model.joblib
 2. unsw_nb15_scaler.joblib
 3. unsw_nb15_encoders.joblib
 4. unsw_nb15_features.jobli